# RAG over radiology reports — validation walkthrough

End-to-end **offline** pipeline: synthetic corpus → chunking → TF-IDF retrieval → grounded generation with citations → abstention check.

⚠️ The corpus is **SYNTHETIC** (invented reports, NOT real patient data). The reports and the example questions are in Italian.

Reference: Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, NeurIPS 2020, arXiv:2005.11401.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[0] / "src"))

from rag_reports.corpus import load_sample_corpus
from rag_reports.rag import RagPipeline
from rag_reports.retriever import build_retriever
from rag_reports.evaluation import evaluate_retrieval, evaluate_abstention

docs = load_sample_corpus()
print(f"Documents in the synthetic corpus: {len(docs)}")
print("All synthetic=True:", all(d['synthetic'] for d in docs))

## 1. TF-IDF retrieval

In [ ]:
retr = build_retriever(docs, dense=False)
for r in retr.search("È presente uno pneumotorace?", k=3):
    print(f"[{r.score:.3f}] {r.chunk.doc_id}: {r.chunk.text[:70]}...")

## 2. RAG pipeline (extractive backend, offline)

In [ ]:
rag = RagPipeline.from_documents(docs, backend="extractive")
ans = rag.answer("Ci sono segni di polmonite al lobo inferiore destro?")
print("Abstained:", ans.abstained)
print("Sources:", ans.source_ids)
print()
print(ans.text)

## 3. Abstention (out-of-corpus query)

In [ ]:
ans = rag.answer("Qual è la capitale della Francia?")
print("Abstained:", ans.abstained)
print("Answer:", ans.text)
assert ans.abstained, "should abstain on an out-of-corpus query"

## 4. Metrics: hit-rate@k and abstention

In [ ]:
retr_eval = evaluate_retrieval(retr, k_values=(1, 3, 5))
print("Hit-rate@k:", {k: round(v, 3) for k, v in retr_eval.hit_rate_at_k.items()})

abst = evaluate_abstention(rag)
print(f"Correct abstentions (out-of-corpus): {abst.n_correct_abstentions}/{abst.n_queries}")